# Merge OpenAlex parts & describe

Loads every `synbio_openalex_PART_*.txt` file produced by `get_synbio_data.ipynb`,
concatenates them, removes duplicates, sanitizes missing values, and shows
summary statistics. The merged file is saved to `../assets/`.

In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
INPUT_DIR  = os.path.join('..', 'assets', 'openalex_data')
OUTPUT_DIR = os.path.join('..', 'assets')

## 1. Load & concatenate all parts

In [ ]:
files = sorted(glob.glob(os.path.join(INPUT_DIR, 'synbio_openalex_PART_*.txt')))
print(f'Found {len(files)} part file(s):')
for f in files:
    print(f'  {os.path.basename(f)}')

parts = [pd.read_csv(f, sep='\t') for f in files]
pubs = pd.concat(parts, ignore_index=True)
print(f'\nTotal rows after concatenation: {len(pubs):,}')

## 2. Deduplicate & sanitize

In [ ]:
before = len(pubs)
pubs.drop_duplicates(subset='id', inplace=True)
pubs.reset_index(drop=True, inplace=True)
print(f'Duplicates removed: {before - len(pubs):,}')
print(f'Unique publications: {len(pubs):,}')

In [ ]:
# Fill NaN in string columns with empty strings
str_cols = pubs.select_dtypes(include='object').columns
pubs[str_cols] = pubs[str_cols].fillna('')

# Fill NaN in numeric columns with 0
num_cols = pubs.select_dtypes(include='number').columns
pubs[num_cols] = pubs[num_cols].fillna(0)

print('NaN counts after sanitization:')
print(pubs.isna().sum())

## 3. Save merged file

In [ ]:
output_path = os.path.join(OUTPUT_DIR, 'synbio_openalex.txt')
pubs.to_csv(output_path, sep='\t', index=False)
print(f'Saved {len(pubs):,} records to {output_path}')

## 4. Summary statistics

In [ ]:
print(f'Shape: {pubs.shape}')
print(f'Columns: {pubs.columns.tolist()}\n')

non_empty = lambda col: (pubs[col].astype(str).str.strip() != '').sum()

for col in ['abstract', 'authors', 'institutions', 'countries', 'concepts', 'referenced_works', 'doi']:
    n = non_empty(col)
    print(f'  {col:20s}  {n:>7,} / {len(pubs):,}  ({n/len(pubs):.1%})')

print(f'\nCitation stats:')
pubs['cited_by_count'].describe()

In [ ]:
# Articles per year
yearly = pubs['publication_year'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(12, 5))
yearly.plot(kind='bar', ax=ax)
ax.set_title('Articles per year')
ax.set_xlabel('Year')
ax.set_ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
def top_values(col: str, sep: str = '; ', n: int = 20):
    """Explode a semicolon-separated column and show the top-n values."""
    vals = pubs[col].str.split(sep).explode().str.strip()
    vals = vals[vals != '']
    return vals.value_counts().head(n)

In [ ]:
# Top 20 countries
top_countries = top_values('countries')

fig, ax = plt.subplots(figsize=(10, 7))
top_countries.plot(kind='barh', ax=ax)
ax.invert_yaxis()
ax.set_xlabel('Number of papers')
ax.set_title('Top 20 countries')
for i, v in enumerate(top_countries):
    ax.text(v, i, f' {v:,}', va='center')
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 institutions
top_inst = top_values('institutions')

fig, ax = plt.subplots(figsize=(10, 7))
top_inst.plot(kind='barh', ax=ax)
ax.invert_yaxis()
ax.set_xlabel('Number of papers')
ax.set_title('Top 20 institutions')
for i, v in enumerate(top_inst):
    ax.text(v, i, f' {v:,}', va='center')
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 concepts
top_concepts = top_values('concepts')

fig, ax = plt.subplots(figsize=(10, 7))
top_concepts.plot(kind='barh', ax=ax)
ax.invert_yaxis()
ax.set_xlabel('Number of papers')
ax.set_title('Top 20 concepts')
for i, v in enumerate(top_concepts):
    ax.text(v, i, f' {v:,}', va='center')
plt.tight_layout()
plt.show()